# Conjugation POC walkthrough

Purpose:

- Build an end-to-end POC for attaching an NHS-containing random copolymer to one enzyme lysine
- Preserve per-monomer residue identity instead of collapsing the polymer to one giant residue
- Hand Pablo an already modified protein/polymer coordinate artifact; Pablo does not generate coordinates
- Keep expensive chemistry/simulation work behind `RUN_*` flags
- Expose seams where abstractions may belong, without turning this into reference docs


## Science-facing workflow map

This walkthrough treats coordinate construction and topology perception as separate steps:

1. Start from a source protein PDB whose hydrogens are canonicalized by PDBFixer plus OpenMM Modeller at a user-configurable pH, then select the target lysine, here chain `A` / `LYS 23` / atom `NZ`.
2. Generate a seeded random 10-mer SBMA/EGPMA/NHS copolymer PDB plus an SDF bond-order sidecar from PolyzyMD's Polymerist boundary.
3. Resolve the NHS-Lys attachment plan from the generated polymer fragment and the source protein.
4. Place the polymer near the target site with Packmol while preserving atom names, residue names, chain IDs, and element fields.
5. Assemble the covalent product with RDKit/PDB graph surgery: remove resolved leaving atoms, rename product residues, add the lysine-polymer amide bond, and write `LINK`/`CONECT` metadata.
6. Validate that chain `C` is still multiple monomer residues (`SBM`/`EGP`/`NHX`) rather than one collapsed `POLY` residue.
7. Optionally run a post-crosslink constrained local minimization through OpenMM/OpenFF/Pablo. Packmol places coordinates and graph surgery creates the covalent product; this physics step is where bad local geometry should be repaired.
8. Stop before Pablo/OpenFF minimization or later Pablo ingestion unless product-state residue definitions are generated from the emitted product PDB/product graph. This notebook records the blocker rather than falling back to RDKit/UFF.
9. Stop before Interchange unless a non-empty explicit charged product template is available for `charge_from_molecules=[template]`.

Important boundary: Pablo custom residue definitions and crosslinks help topology perception. They cannot repair bad PDB metadata, generate coordinates, infer missing elements, rescue a collapsed `POLY` residue representation, or replace explicit product-state charge templates.

## 0. Setup

Use `pixi run -e build jupyter lab ...` for heavy cells. All heavy flags are `False` by default.


In [ ]:
from __future__ import annotations

import ast
import importlib.util
import os
import sys
from pathlib import Path
from pprint import pprint

TRUE_ENV_VALUES = {"1", "true", "t", "yes", "y", "on"}


def env_flag(name: str, default: bool = False) -> bool:
    """Return an opt-in boolean flag from the environment."""
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in TRUE_ENV_VALUES


RUN_POLYMERIST = env_flag("RUN_POLYMERIST", default=False)
RUN_PACKMOL = env_flag("RUN_PACKMOL", default=False)
RUN_GENERATED_PRODUCT = env_flag("RUN_GENERATED_PRODUCT", default=False)
RUN_LOCAL_MINIMIZATION = env_flag("RUN_LOCAL_MINIMIZATION", default=False)
RUN_PABLO = env_flag("RUN_PABLO", default=False)
RUN_PABLO_SMOKE = env_flag("RUN_PABLO_SMOKE", default=False)
RUN_PABLO_REACTION = env_flag("RUN_PABLO_REACTION", default=False)
RUN_CHARGE_PROTOCOL = env_flag("RUN_CHARGE_PROTOCOL", default=False)
RUN_PARAMETERIZATION = env_flag("RUN_PARAMETERIZATION", default=False)
RUN_SYSTEM_WORKFLOW = env_flag("RUN_SYSTEM_WORKFLOW", default=False)
RUN_MARIMO_POC = False # env_flag("RUN_MARIMO_POC", default=False)
RUN_INTERCHANGE_HANDOFF = env_flag("RUN_INTERCHANGE_HANDOFF", default=False)
PROTEIN_CANONICALIZATION_PH = float(os.environ.get("PROTEIN_CANONICALIZATION_PH", "7.0"))


def find_repo_root(start: Path | None = None) -> Path:
    """Find the PolyzyMD repository root from a starting path."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "polyzymd").exists():
            return candidate
    raise RuntimeError(f"Could not find PolyzyMD repo root from {current}")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
POC_DIR = SRC_DIR / "polyzymd" / "builders" / "conjugation" / "poc"
CONJ_DIR = POC_DIR.parent
SOURCE_PROTEIN_PDB = POC_DIR / "data" / "NH3_terminal_His_proton_updated.pdb"
LEGACY_CONJUGATE_PDB = POC_DIR / "1LYZ_conj.pdb"
CANONICAL_PROTEIN_PDB = (
    POC_DIR / "output" / f"source_protein_canonical_pH{PROTEIN_CANONICALIZATION_PH:g}.pdb"
)
PROTEIN_PDB = SOURCE_PROTEIN_PDB

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Repo root: {REPO_ROOT}")
print(f"POC dir: {POC_DIR}")
print(f"Source-protein hydrogen canonicalization pH: {PROTEIN_CANONICALIZATION_PH:g}")

## 0.1 Source-Protein Hydrogen Canonicalization

Before any conjugation or Pablo ingestion, this walkthrough canonicalizes the source protein hydrogens with PDBFixer followed by OpenMM `Modeller.delete()` for existing hydrogens and `Modeller.addHydrogens(ForceField('amber14/protein.ff14SB.xml'), pH=...)`. The pH is user-configurable through `PROTEIN_CANONICALIZATION_PH` and defaults to `7.0`. Chain IDs and residue numbers are preserved with `keepIds=True` when OpenMM can preserve them.

The generated product below uses this canonicalized source protein, not the raw input PDB, so Pablo sees OpenMM-compatible protein hydrogen names before product-state residue definitions are applied.


In [ ]:
from polyzymd.builders.conjugation.protein_preparation import (
    ProteinCanonicalizationSettings,
    canonicalize_protein_hydrogens,
)

canonicalization_result = canonicalize_protein_hydrogens(
    SOURCE_PROTEIN_PDB,
    CANONICAL_PROTEIN_PDB,
    settings=ProteinCanonicalizationSettings(ph=PROTEIN_CANONICALIZATION_PH),
)
canonicalization_summary_path = canonicalization_result.save()
PROTEIN_PDB = canonicalization_result.output_path
pprint(canonicalization_result.model_dump(mode="json"))
print(f"Canonicalization summary: {canonicalization_summary_path}")
print(f"Canonicalized source protein selected for conjugation: {PROTEIN_PDB}")


## 1. Inputs and PDB hygiene baseline

- Inspect file assets without importing the Marimo app
- Source PDB is the positive Pablo baseline and the only protein input used for the happy path
- Legacy `1LYZ_conj.pdb` is retained as a negative-control diagnostic for bad assembled metadata, not as a working product
- Custom Pablo residue definitions can describe noncanonical chemistry only after the PDB has valid chain IDs, residue names, atom names, element fields, and linkage metadata


In [ ]:
def describe_path(path: Path) -> dict[str, object]:
    """Return a compact description of a file-system asset."""
    return {
        "name": path.name,
        "kind": "dir" if path.is_dir() else path.suffix or "file",
        "size_kb": None if path.is_dir() else round(path.stat().st_size / 1024, 1),
    }


assets = [describe_path(p) for p in sorted(POC_DIR.iterdir(), key=lambda p: p.name.lower())]
pprint(assets)

print("\nReusable conjugation modules:")
for path_ in sorted(CONJ_DIR.glob("*.py")):
    print(f"- {path_.relative_to(REPO_ROOT)}")


In [ ]:
STANDARD_PDB_RESIDUES = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "HID", "HIE",
    "HIP", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR",
    "VAL", "ACE", "NME", "HOH", "WAT", "TIP", "SOL", "NA", "CL", "K", "MG", "CA",
    "ZN",
}


def inspect_pdb_metadata(path: Path) -> dict[str, object]:
    """Summarize fixed-width PDB fields that matter to Pablo ingestion."""
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    atom_lines = [line for line in lines if line.startswith(("ATOM", "HETATM"))]
    residues = {
        (line[21:22].strip() or "<blank>", line[22:26].strip(), line[17:20].strip())
        for line in atom_lines
    }
    residue_names = sorted({line[17:20].strip() for line in atom_lines})
    return {
        "file": path.name,
        "atoms": len(atom_lines),
        "residues": len(residues),
        "chains": sorted({line[21:22].strip() or "<blank>" for line in atom_lines}),
        "blank_chain_atoms": sum(1 for line in atom_lines if not line[21:22].strip()),
        "missing_elements": sum(1 for line in atom_lines if len(line) < 78 or not line[76:78].strip()),
        "custom_resnames": [name for name in residue_names if name not in STANDARD_PDB_RESIDUES],
        "link_records": sum(line.startswith("LINK") for line in lines),
        "conect_records": sum(line.startswith("CONECT") for line in lines),
    }


pdb_baseline_table = [
    {"role": "raw source reference", **inspect_pdb_metadata(SOURCE_PROTEIN_PDB)},
    {"role": "canonicalized source used for conjugation", **inspect_pdb_metadata(PROTEIN_PDB)},
    {"role": "legacy negative control", **inspect_pdb_metadata(LEGACY_CONJUGATE_PDB)},
]
pprint(pdb_baseline_table)


Short note: custom Pablo residue definitions cannot fix blank chains, missing element fields, absent connectivity, or residue records collapsed to `POLY`. The next generated product should start from the source PDB and avoid the legacy artifact's metadata failures.


In [ ]:
def pablo_smoke_load(path: Path) -> dict[str, object]:
    """Try a guarded Pablo load and return a compact status dictionary."""
    from openff.pablo import topology_from_pdb

    try:
        topology = topology_from_pdb(path)
    except Exception as exc:
        message = str(exc).splitlines()[0] if str(exc).splitlines() else ""
        return {"ok": False, "error": type(exc).__name__, "message": message}
    return {"ok": True, "n_atoms": topology.n_atoms, "n_molecules": topology.n_molecules}


if RUN_PABLO_SMOKE:
    pprint(
        {
            "canonical_source_expected_load": pablo_smoke_load(PROTEIN_PDB),
            "raw_source_reference": pablo_smoke_load(SOURCE_PROTEIN_PDB),
            "legacy_expected_fail": pablo_smoke_load(LEGACY_CONJUGATE_PDB),
        }
    )
else:
    print("Skipping Pablo smoke check. Set RUN_PABLO_SMOKE = True to run it.")


## 2. Protein site selection

Old POC target: chain A, `LYS 23`, atom `NZ`.


In [ ]:
def parse_pdb_atoms(path: Path) -> list[dict[str, object]]:
    """Parse ATOM/HETATM records needed for target-site inspection."""
    atoms: list[dict[str, object]] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            atoms.append(
                {
                    "index": len(atoms),
                    "serial": int(line[6:11]),
                    "name": line[12:16].strip(),
                    "resname": line[17:20].strip(),
                    "chain": line[21].strip() or " ",
                    "resid": int(line[22:26]),
                    "x": float(line[30:38]),
                    "y": float(line[38:46]),
                    "z": float(line[46:54]),
                    "element": line[76:78].strip() or line[12:16].strip()[0],
                }
            )
    return atoms


atoms = parse_pdb_atoms(PROTEIN_PDB)
residues = {(atom["chain"], atom["resid"], atom["resname"]) for atom in atoms}
lysines = sorted({(atom["chain"], atom["resid"]) for atom in atoms if atom["resname"] == "LYS"})
TARGET_LYS_RESIDS = (23,)

print(f"Protein atoms: {len(atoms)}")
print(f"Residues: {len(residues)}")
print(f"LYS residues: {lysines}")
for resid in TARGET_LYS_RESIDS:
    site_atoms = [
        atom
        for atom in atoms
        if atom["chain"] == "A" and atom["resid"] == resid and atom["resname"] == "LYS"
    ]
    print(f"\nChain A LYS {resid}: {len(site_atoms)} atoms")
    pprint([atom for atom in site_atoms if atom["name"] in {"CE", "NZ"}])


## 3. Moiety/polymer generation

POC chemistry assumptions:

- Seeded random SBMA/EGPMA/NHS copolymer; deterministic `ACB` smoke recipe remains a small diagnostic
- NHS ester reacts with lysine `NZ`
- Product PDB residue labels: `LYX` for the reacted lysine and `NHX` for the reacted NHS-derived monomer
- These residue labels are not Pablo product-state residue definitions by themselves

In [ ]:
def marimo_literal_constants() -> dict[str, object]:
    """Read simple uppercase literal constants from the old Marimo POC."""
    tree = ast.parse((POC_DIR / "conjugation_poc.py").read_text(encoding="utf-8"))
    values: dict[str, object] = {}
    for node in ast.walk(tree):
        if not isinstance(node, ast.Assign):
            continue
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id.isupper():
                try:
                    values[target.id] = ast.literal_eval(node.value)
                except Exception:
                    pass
    return values


literal_assignments = marimo_literal_constants()
for key in [
    "MONOMER_NAMES", "MONOMER_PROBABILITIES", "RESIDUE_NAMES", "NHS_LABEL",
    "CONJUGATION_SITE_RESIDS", "CONJUGATED_LENGTH", "PROTEIN_FF", "SMALL_MOL_FF",
]:
    print(f"{key} = {literal_assignments.get(key)!r}")


In [ ]:
from polyzymd.builders.conjugation.linkers import NhsLysModifierLinker
from polyzymd.builders.conjugation.polymer_recipe import (
    sbma_egpma_nhs_recipe,
    sbma_nhs_egpma_acb_recipe,
)

linker = NhsLysModifierLinker(target_chain="A", target_residue_number=23)
long_recipe = sbma_egpma_nhs_recipe(length=10, seed=42)
small_recipe = sbma_nhs_egpma_acb_recipe()

pprint(
    {
        "linker_target": f"{linker.target_chain}:LYS{linker.target_residue_number}:NZ",
        "lys_product_resname": linker.lysine_target_resname,
        "modifier_product_resname": linker.modifier_target_resname,
        "long_sequence": long_recipe.generate_sequence(),
        "smoke_sequence": small_recipe.generate_sequence(),
        "smoke_reactive_index": small_recipe.effective_reactive_index,
    }
)


In [ ]:
def available(module: str) -> bool:
    """Return whether an optional module can be imported in this kernel."""
    return importlib.util.find_spec(module) is not None


for module in ["polymerist", "mbuild", "rdkit", "openff.toolkit"]:
    print(f"{module:18s}", "available" if available(module) else "missing")

polymerist_results = {}
generated_fragments = {}

if RUN_POLYMERIST:
    from polyzymd.builders.conjugation.polymer_recipe import generate_polymerist_smoke_polymer
    from polyzymd.builders.conjugation.polymerist_pdb import (
        generated_fragment_from_polymerist_pdb,
    )

    polymerist_runs = {
        "deterministic_smoke_3mer": small_recipe,
        "seeded_random_10mer": long_recipe,
    }
    for label, recipe in polymerist_runs.items():
        cache_dir = POC_DIR / "output" / f"walkthrough-polymerist-{label}"
        generation = generate_polymerist_smoke_polymer(
            recipe,
            cache_dir,
            max_retries=3,
            energy_minimize=True,
        )
        polymerist_results[label] = generation
        print(f"\n{label}")
        pprint(generation.model_dump(mode="json", exclude={"rdkit_mol"}))

        if generation.pdb_path is not None:
            fragment = generated_fragment_from_polymerist_pdb(
                generation.pdb_path,
                recipe=recipe,
                sequence=generation.sequence,
                name=label,
            )
            generated_fragments[label] = fragment
            print("Generated fragment checkpoint:")
            pprint(
                {
                    "atom_count": len(fragment.atoms),
                    "residue_count": len(fragment.residues),
                    "sequence": fragment.sequence,
                    "reactive_atom_name": fragment.reactive_atom_name,
                    "leaving_atom_names": fragment.leaving_atom_names,
                    "residue_names": [res.residue_name for res in fragment.residues],
                }
            )
else:
    print("Skipping Polymerist generation. Set RUN_POLYMERIST = True to run it.")
    print("Expected branches: deterministic smoke 3-mer and seeded random 10-mer.")


## 4. Placement near target residue

Packmol concept:

- Fixed protein sterics, with target/reactive leaving atoms handled specially
- Movable retained modifier atoms
- Reactive atom constrained near lysine `NZ`

Code smell note: Packmol constraints know chemistry-specific reactive/leaving atoms; this likely wants a generic resolved attachment plan.


In [ ]:
try:
    from polyzymd.utils.packmol import build_packmol_input, run_packmol
except Exception as exc:
    build_packmol_input = None
    print(f"Packmol utility import failed: {exc}")

if build_packmol_input is not None:
    work = POC_DIR / "output" / "walkthrough-packmol-dryrun"
    work.mkdir(parents=True, exist_ok=True)
    protein_stub = work / "protein_fixed_sterics_stub.pdb"
    modifier_stub = work / "modifier_retained_stub.pdb"
    protein_stub.write_text(
        "HETATM    1 C1   UNK A   1      10.000  10.000  10.000  1.00  0.00           C\nEND\n"
    )
    modifier_stub.write_text(
        "HETATM    1 C1   UNK C   1      12.000  10.000  10.000  1.00  0.00           C\nEND\n"
    )
    input_text = build_packmol_input(
        molecule_pdb_paths=[str(modifier_stub)],
        molecule_counts=[1],
        box_size_angstrom=[40.0, 40.0, 40.0],
        tolerance_angstrom=2.0,
        solute_pdb_path=str(protein_stub),
        use_pbc=False,
        movebadrandom=True,
        nloop=500,
        structure_extra_lines=[
            ["atoms 1", "inside sphere 10.000 10.000 10.000 5.0", "end atoms"]
        ],
    )
    print(input_text)
    if RUN_PACKMOL:
        print(run_packmol(input_text, work))
    else:
        print("Skipping Packmol execution. Set RUN_PACKMOL = True to run it.")


## 5. Generated coordinate product and validation

The main POC path in this notebook is the guarded cell below. It generates the seeded NHS-containing random copolymer, resolves the attachment plan, places the polymer near `LYS 23 NZ`, writes the modified crosslinked PDB, and validates residue identity.

The cell is heavy and stays off by default. When it is not run, the notebook truthfully stops before the assembled-product checkpoint instead of selecting a stale pre-existing PDB as the happy path.

Assembled product expectations:

- Chain `A` remains the protein chain
- Chain `C` contains the polymer as multiple monomer residues (`SBM`, `EGP`, `NHS`/`NHX`), not one collapsed `POLY` residue
- Target lysine `LYS 23` is renamed `LYX` after reaction
- The reactive NHS-derived monomer is renamed `NHX`
- NHS leaving atoms are removed from the polymer side
- The resolved lysine reactive H atoms are removed from `NZ`
- A `LINK` and/or `CONECT` record advertises the `LYX:NZ` to `NHX:<resolved acyl carbon>` bond

In [ ]:
from polyzymd.builders.conjugation.contracts import placed_fragment_from_resolved_plan
from polyzymd.builders.conjugation.pdb_assembly import (
    CrosslinkedPdbAssemblyOptions,
    write_crosslinked_pdb,
)
from polyzymd.builders.conjugation.placement import (
    PackmolModifierPlacementSettings,
    place_modifier_with_packmol,
)
from polyzymd.builders.conjugation.polymer_recipe import generate_polymerist_smoke_polymer
from polyzymd.builders.conjugation.polymerist_pdb import generated_fragment_from_polymerist_pdb

assembled_like = None
selected_product_pdb = None
generated_product = None
generated_product_generation = None
generated_product_fragment = None
resolved_plan = None
placement_result = None
assembly_result = None

if RUN_GENERATED_PRODUCT:
    work = POC_DIR / "output" / "walkthrough-generated-product"
    work.mkdir(parents=True, exist_ok=True)

    generated_product_generation = generate_polymerist_smoke_polymer(
        sbma_egpma_nhs_recipe(length=10, seed=42),
        work / "polymerist-cache",
        force_regenerate=True,
        max_retries=3,
        energy_minimize=True,
    )
    if generated_product_generation.pdb_path is None:
        raise RuntimeError("Polymerist did not produce a seeded random 10-mer PDB")

    generated_product_fragment = generated_fragment_from_polymerist_pdb(
        generated_product_generation.pdb_path,
        recipe=long_recipe,
        sequence=generated_product_generation.sequence,
        name="seeded_random_10mer",
    )
    resolved_plan = linker.resolve_plan(PROTEIN_PDB, generated_product_fragment)

    placement_result = place_modifier_with_packmol(
        PROTEIN_PDB,
        generated_product_fragment,
        linker,
        work,
        settings=PackmolModifierPlacementSettings(nloop=500),
    )
    placed_modifier = placed_fragment_from_resolved_plan(
        placement_result.placed_modifier,
        resolved_plan,
    )

    assembled_like = work / "seeded_random_10mer_crosslinked.pdb"
    assembly_result = write_crosslinked_pdb(
        PROTEIN_PDB,
        placed_modifier,
        resolved_plan.to_nhs_lys_pdb_attachment(),
        assembled_like,
        CrosslinkedPdbAssemblyOptions(),
    )

    generated_product = {
        "source_protein_pdb": str(PROTEIN_PDB),
        "source_protein_canonicalization_ph": PROTEIN_CANONICALIZATION_PH,
        "sequence": generated_product_generation.sequence,
        "polymer_pdb": str(generated_product_generation.pdb_path),
        "crosslinked_pdb": str(assembled_like),
        "reactive_residue": {
            "chain": resolved_plan.modifier_link_atom.chain_id,
            "resname": resolved_plan.modifier_link_atom.residue_name,
            "resid": resolved_plan.modifier_link_atom.residue_number,
            "atom": resolved_plan.modifier_link_atom.atom_name,
        },
        "protein_leaving_atoms": [atom.atom_name for atom in resolved_plan.protein_leaving_atoms],
        "modifier_leaving_atoms": [atom.atom_name for atom in resolved_plan.modifier_leaving_atoms],
        "assembly": assembly_result.model_dump(mode="json"),
    }
    pprint(generated_product)
else:
    print("Skipping generated product construction. Set RUN_GENERATED_PRODUCT = True to run it.")
    print("No stale pre-existing assembled PDB is selected as the happy path.")

if assembled_like is not None:
    selected_product_pdb = assembled_like
    lines = assembled_like.read_text(encoding="utf-8", errors="replace").splitlines()
    atom_lines = [line for line in lines if line.startswith(("ATOM", "HETATM"))]
    conect_lines = [line for line in lines if line.startswith("CONECT")]
    resnames = sorted({line[17:20].strip() for line in atom_lines})
    pprint(
        {
            "role": "generated assembled product",
            "atoms": len(atom_lines),
            "chains": sorted({line[21].strip() or "<blank>" for line in atom_lines}),
            "product_resname_hints": [
                name for name in resnames if name in {"SBM", "EGP", "NHS", "NHX", "LYX", "PLL", "PME"}
            ],
            "conect_records": len(conect_lines),
            "first_conect": conect_lines[:3],
        }
    )
else:
    print("Generated assembled product unavailable; this notebook currently stops before Pablo.")

print("\nLegacy negative-control metadata only:")
pprint(inspect_pdb_metadata(LEGACY_CONJUGATE_PDB))

In [ ]:
def validate_assembled_product(path: Path | None) -> dict[str, object]:
    """Return pass/fail checkpoints for an assembled conjugate PDB."""
    if path is None:
        return {"available": False, "message": "No assembled product PDB selected"}

    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    atom_lines = [line for line in lines if line.startswith(("ATOM", "HETATM"))]
    residues_by_chain = {}
    resnames = set()
    for line in atom_lines:
        chain = line[21:22].strip() or "<blank>"
        key = (line[22:26].strip(), line[26:27].strip(), line[17:20].strip())
        residues_by_chain.setdefault(chain, set()).add(key)
        resnames.add(line[17:20].strip())

    metadata = inspect_pdb_metadata(path)
    chain_c_residues = residues_by_chain.get("C", set())
    link_or_conect = metadata["link_records"] + metadata["conect_records"]
    checks = {
        "no_blank_chains": metadata["blank_chain_atoms"] == 0,
        "no_missing_elements": metadata["missing_elements"] == 0,
        "protein_chain_A_present": "A" in residues_by_chain,
        "polymer_chain_C_present": "C" in residues_by_chain,
        "chain_C_has_multiple_residues": len(chain_c_residues) > 1,
        "chain_C_not_collapsed_to_POLY": not any(resname == "POLY" for *_rest, resname in chain_c_residues),
        "LYX_present": "LYX" in resnames,
        "NHX_present": "NHX" in resnames,
        "linkage_metadata_present": link_or_conect > 0,
    }
    return {
        "available": True,
        "file": path.name,
        "checks": checks,
        "failed": [name for name, ok in checks.items() if not ok],
        "chain_C_residue_count": len(chain_c_residues),
        "chain_C_residue_names": sorted({resname for *_rest, resname in chain_c_residues}),
        "link_records": metadata["link_records"],
        "conect_records": metadata["conect_records"],
    }


assembled_product_checkpoint = validate_assembled_product(assembled_like)
pprint(assembled_product_checkpoint)


## 6. Post-crosslink local minimization

Packmol placement and PDB graph surgery create an already crosslinked coordinate artifact while preserving PDB metadata. They do not perform force-field relaxation. The guarded cell below attempts the intended physics repair: Pablo ingests the already modified PDB, OpenFF/OpenMM parameterize it with ff14SB plus OpenFF coverage, nonparticipating protein atoms are strongly restrained, and only coordinate columns are replaced in the relaxed PDB.

The local minimization cell builds product-state Pablo residue definitions from the emitted PDB/product graph and passes the prebuilt library to Pablo. That avoids the previous `with_crosslink(... leaving_atoms=((), ()))` path while preserving `LYX`, `NHX`, and per-monomer chain-C residue identities. If Pablo/OpenFF cannot parameterize the product-state `LYX`/polymer residue graph, the notebook records the exact blocker and does not fall back to RDKit/UFF. Downstream cells use `selected_product_pdb`, which points to the relaxed PDB only after a successful minimization.

In [ ]:
from polyzymd.builders.conjugation.local_minimization import (
    CrosslinkAtomSelector,
    LocalMinimizationSettings,
    analyze_crosslink_geometry,
    product_state_pablo_crosslink_requirement,
    run_post_crosslink_local_minimization,
)
from polyzymd.builders.conjugation.product_pablo import build_product_state_pablo_library

local_minimization_result = None
local_minimization_crosslink_requirement = None
product_state_pablo_library = None
relaxed_product_pdb = None
selected_product_pdb = assembled_like
local_minimization_settings = None


def local_minimization_settings_for_product(path: Path) -> LocalMinimizationSettings:
    """Build selector settings from emitted product atom identities, not stale serials."""
    product_atoms = parse_pdb_atoms(path)

    def unique_atom(resname: str, atom_name: str, *, same_residue_as=None):
        matches = [
            atom
            for atom in product_atoms
            if atom["resname"] == resname and atom["name"] == atom_name
        ]
        if same_residue_as is not None:
            matches = [
                atom
                for atom in matches
                if atom["chain"] == same_residue_as["chain"]
                and atom["resid"] == same_residue_as["resid"]
            ]
        if len(matches) != 1:
            raise RuntimeError(f"Expected exactly one {resname}:{atom_name}, found {len(matches)}")
        return matches[0]

    nz_atom = unique_atom("LYX", "NZ")
    c047_atom = unique_atom("NHX", "C047")
    o020_atom = unique_atom("NHX", "O020", same_residue_as=c047_atom)

    def selector(atom) -> CrosslinkAtomSelector:
        return CrosslinkAtomSelector(
            serial=None,
            chain_id=atom["chain"],
            residue_name=atom["resname"],
            residue_number=atom["resid"],
            atom_name=atom["name"],
        )

    return LocalMinimizationSettings(
        nz_selector=selector(nz_atom),
        c047_selector=selector(c047_atom),
        o020_selector=selector(o020_atom),
    )

if assembled_like is None:
    print("No generated product is available for local minimization.")
else:
    local_minimization_settings = local_minimization_settings_for_product(assembled_like)
    raw_geometry = analyze_crosslink_geometry(assembled_like, settings=local_minimization_settings)
    print("Raw crosslinked geometry:")
    pprint(raw_geometry.model_dump())
    if resolved_plan is not None:
        local_minimization_crosslink_requirement = product_state_pablo_crosslink_requirement(
            assembled_like,
            settings=local_minimization_settings,
            resolved_plan=resolved_plan,
        )
        print("Product-state Pablo crosslink for local minimization:")
        pprint(local_minimization_crosslink_requirement.model_dump())
        if generated_product_fragment is not None:
            product_state_pablo_library = build_product_state_pablo_library(
                product_pdb=assembled_like,
                source_protein_pdb=PROTEIN_PDB,
                polymer_sdf=getattr(generated_product_generation, "sdf_path", None),
                generated_fragment=generated_product_fragment,
                resolved_plan=resolved_plan,
            )
            print("Product-state Pablo residue library summaries:")
            pprint(product_state_pablo_library.model_dump(mode="json"))

    if RUN_LOCAL_MINIMIZATION:
        local_minimization_result = run_post_crosslink_local_minimization(
            assembled_like,
            assembled_like.parent,
            settings=local_minimization_settings,
            pablo_crosslink_requirement=local_minimization_crosslink_requirement,
            product_state_pablo_library=product_state_pablo_library,
        )
        summary = local_minimization_result.model_dump(
            mode="json",
            exclude={"blocker_traceback"},
        )
        print("Local minimization result:")
        pprint(summary)
        if local_minimization_result.success and local_minimization_result.relaxed_pdb_path:
            relaxed_product_pdb = local_minimization_result.relaxed_pdb_path
            selected_product_pdb = relaxed_product_pdb
    else:
        print("Skipping OpenMM/OpenFF/Pablo local minimization.")
        print("Set RUN_LOCAL_MINIMIZATION = True to attempt the physics relaxation.")

    if relaxed_product_pdb is None:
        print("Downstream cells will continue to use the raw assembled PDB.")
    else:
        print(f"Downstream cells will use relaxed PDB: {relaxed_product_pdb}")


## 7. Charges and force-field handoff

Charge protocol transcription:

- Extract ff14SB charges from unmodified protein
- Build an explicit charged product template for the assembled protein/polymer product before Interchange
- Product-state charge demos below are proxies only; they are not accepted as handoff templates for the generated conjugate
- Call Interchange only with `charge_from_molecules=[template]` where `template` is non-empty and corresponds to the product topology
- Treat this as a handoff smoke check only; production charge readiness still requires validated product-state charge templates and force-field coverage

Code smell note: hybrid charge patching is against-the-grain and should become a tested charge-template builder rather than notebook glue.

In [ ]:
from polyzymd.builders.conjugation.parameterization import InterchangeParameterizationSettings

optional_modules = [
    "openff.toolkit",
    "openff.interchange",
    "openff.pablo",
    "openff.toolkit.utils.nagl_wrapper",
    "openmm",
    "rdkit",
    "polymerist",
]
pprint({name: available(name) for name in optional_modules})
print("Default force fields:", InterchangeParameterizationSettings().force_field_names)
print("POC protein FF:", literal_assignments.get("PROTEIN_FF"))
print("POC small-molecule FF:", literal_assignments.get("SMALL_MOL_FF"))


In [ ]:
protein_charge_arr = None
protein_topology = None

if RUN_CHARGE_PROTOCOL or RUN_PARAMETERIZATION:
    try:
        import logging as _logging

        import numpy as np
        from openff.toolkit import ForceField, Topology
        from openff.units import unit

        protein_ff_name = literal_assignments.get("PROTEIN_FF") or "ff14sb_off_impropers_0.0.4.offxml"
        protein_topology = Topology.from_pdb(str(PROTEIN_PDB))
        protein_ff = ForceField(protein_ff_name)
        nonbonded_logger = _logging.getLogger("openff.interchange.smirnoff._nonbonded")
        previous_level = nonbonded_logger.level
        nonbonded_logger.setLevel(_logging.WARNING)
        try:
            protein_interchange = protein_ff.create_interchange(protein_topology)
        finally:
            nonbonded_logger.setLevel(previous_level)

        charges = protein_interchange["Electrostatics"].charges
        protein_charge_arr = np.array(
            [
                charges[key].m_as(unit.elementary_charge)
                for key in sorted(charges.keys(), key=lambda key: key.atom_indices[0])
            ]
        )
        print(f"Extracted ff14SB charges: {len(protein_charge_arr)} atoms")
        print(f"Total charge: {protein_charge_arr.sum():+.4f} e")
    except Exception as exc:
        protein_charge_arr = None
        protein_topology = None
        print(f"Skipping ff14SB extraction: {type(exc).__name__}: {exc}")
else:
    print("Skipping ff14SB extraction. Set RUN_CHARGE_PROTOCOL = True to run it.")


In [ ]:
amide_charge_proxy_demo = None
charged_product_template = None

if RUN_CHARGE_PROTOCOL:
    try:
        from openff.toolkit import Molecule
        from openff.units import unit

        amide_proxy = Molecule.from_smiles("CC(=O)NCC")
        amide_proxy.generate_conformers(n_conformers=1)
        try:
            from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

            amide_proxy.assign_partial_charges(
                partial_charge_method="openff-gnn-am1bcc-0.1.0-rc.3.pt",
                toolkit_registry=NAGLToolkitWrapper(),
            )
            method = "NAGL/AshGC"
        except (ImportError, ValueError, RuntimeError) as exc:
            print(f"NAGL/AshGC unavailable or failed ({exc}); using AM1-BCC proxy fallback.")
            amide_proxy.assign_partial_charges(partial_charge_method="am1bcc")
            method = "AM1-BCC proxy fallback"
        amide_charge_proxy_demo = {
            "method": method,
            "n_atoms": amide_proxy.n_atoms,
            "net_charge_e": float(amide_proxy.total_charge.m_as(unit.elementary_charge)),
            "limitation": "This small amide proxy is not a charged template for the generated conjugate.",
        }
        pprint(amide_charge_proxy_demo)
    except Exception as exc:
        print(f"Charge proxy demo failed: {type(exc).__name__}: {exc}")
else:
    print("Skipping charge proxy demo. Set RUN_CHARGE_PROTOCOL = True to run it.")
    print("charged_product_template remains None until an exact product template builder exists.")

In [ ]:
def merge_hybrid_charge_vector(
    protein_ff14sb_charges,
    retained_protein_old_indices,
    product_fragment_charges,
    junction_charge_patches=None,
):
    """Merge protein, product-fragment, and junction-patch charges."""
    merged = [float(protein_ff14sb_charges[i]) for i in retained_protein_old_indices]
    merged.extend(float(charge) for charge in product_fragment_charges)
    for merged_atom_index, charge in (junction_charge_patches or {}).items():
        merged[merged_atom_index] = float(charge)
    return merged


print("Interchange handoff requirement:")
print('ff = ForceField("ff14sb_off_impropers_0.0.4.offxml", "openff-2.0.0.offxml", ...)')
print("interchange = ff.create_interchange(topology, charge_from_molecules=[charged_product_template])")
print("This notebook refuses to call Interchange with an empty charge template list or with a proxy molecule.")

## 8. Custom Pablo Residue Definitions

Custom Pablo residue definitions are topology templates, not FF parameters or charges. They must match the already assembled PDB exactly.

Current notebook status: the generated-product path can produce and validate an assembled product PDB, but product-state Pablo residue-definition generation from that PDB/product graph is not implemented here. Pablo execution is therefore blocked unless `product_state_pablo_library` is supplied by a real generator.

Requirements before enabling Pablo:

- Build residue definitions from product-state residue graphs with exact PDB atom names
- Preserve per-monomer identity: define `LYX` for the reacted lysine and `NHX` only for the reacted NHS-derived monomer; keep the other polymer monomers as their own residues
- Derive the Pablo crosslink from the resolved attachment plan, not fallback atom names
- Anti-pattern: do not model the whole chain C polymer as one `POLY` or `KPOL` residue just to make Pablo accept one giant template
- Anti-pattern: do not use placeholder LYX/NHX molecules unrelated to the emitted product PDB

In [ ]:
NHS_LYS_REACTION_SMARTS = {
    "reactants": [
        "[CH2:1][N:2]([H:3])[H]",
        "[C:4](=[O:5])[O:6][N:7]1[C:8](=[O:9])[C:10][C:11][C:12]1=[O:13]",
    ],
    "products": [
        "[C:1][N:2][C:4](=[O:5])",
        "[H:3][O:6][N:7]1[C:8](=[O:9])[C:10][C:11][C:12]1=[O:13]",
    ],
}


def pablo_crosslink_spec_from_plan(plan, product_pdb: Path | None) -> dict[str, object] | None:
    """Return product-state Pablo crosslink metadata for the emitted PDB."""
    if plan is None or product_pdb is None:
        return None
    requirement = product_state_pablo_crosslink_requirement(
        product_pdb,
        resolved_plan=plan,
    )
    return {
        "residues": requirement.residues,
        "linking_atoms": requirement.linking_atoms,
        "leaving_atoms": requirement.leaving_atoms,
        "bond_order": requirement.bond_order,
    }


pablo_crosslink_spec = pablo_crosslink_spec_from_plan(resolved_plan, selected_product_pdb)
pprint(
    {
        "reaction_smarts_validation": NHS_LYS_REACTION_SMARTS,
        "pablo_crosslink_spec": pablo_crosslink_spec,
        "source": "resolved attachment plan" if pablo_crosslink_spec else None,
        "note": (
            "Pablo crosslink metadata is unavailable until RUN_GENERATED_PRODUCT builds "
            "a resolved attachment plan. No fallback atom names are used."
            if pablo_crosslink_spec is None
            else (
                "Crosslink metadata was derived from the generated product's resolved plan; "
                "leaving_atoms are empty for the already-modified product PDB."
            )
        ),
    }
)

### Generic Reaction Roles and Retired Pablo React Exploration

This section exposes the intended moving parts before they are hidden behind a production abstraction. PolyzyMD owns concrete atom identity resolution from atom-mapped mechanism metadata: site/moiety participants, linking atoms, leaving atoms, retained atoms, geometry anchors, and mapped bond changes. Pablo is then explored at the residue-definition level with two product residue names, `LYX` and `NHX`, preserving residue identity instead of collapsing the polymer into one residue.

The role derivation originally demonstrated the contract shape for mapped reaction participants and optionally explored Pablo `ResidueDefinition.react()`. That exploratory diagnostic was retired with the removed prototype helper module. This historical notebook is preserved as a reference asset; use the production product-state Pablo/OpenFF path below for maintained behavior.

The original code cell here imported a prototype Pablo reaction diagnostic helper and built placeholder mapped reaction roles for an opt-in `ResidueDefinition.react()` experiment. That helper was removed during the public API cleanup, so the executable diagnostic has been retired instead of kept as stale code.

For maintained conjugation behavior, use the public conjugation API or the product-state Pablo library workflow in the following sections.

In [ ]:
from polyzymd.builders.conjugation.product_pablo import build_product_state_pablo_library

if RUN_PABLO_SMOKE:
    try:
        if product_state_pablo_library is None:
            if selected_product_pdb is None or resolved_plan is None:
                raise RuntimeError("Need a generated crosslinked PDB and resolved plan first")
            product_state_pablo_library = build_product_state_pablo_library(
                product_pdb=selected_product_pdb,
                source_protein_pdb=SOURCE_PROTEIN_PDB,
                polymer_sdf=getattr(generated_product_generation, "sdf_path", None),
                generated_fragment=generated_product_fragment,
                resolved_plan=resolved_plan,
            )
        print("Product-state Pablo library is available:")
        pprint(product_state_pablo_library.model_dump(mode="json"))
    except Exception as exc:
        product_state_pablo_library = None
        print(f"Pablo custom-library setup failed: {type(exc).__name__}: {exc}")
else:
    print("Skipping product-state Pablo library build. Set RUN_PABLO_SMOKE = True after implementing it.")

In [ ]:
def inspect_selected_pdb_for_pablo(pdb_path: Path | None) -> dict[str, object]:
    """Return a compact Pablo preflight summary for an assembled PDB."""
    if pdb_path is None:
        return {"available": False, "message": "No generated crosslinked PDB selected"}
    summary = inspect_pdb_metadata(pdb_path)
    warnings = []
    if summary["custom_resnames"]:
        warnings.append(f"Custom definitions needed: {summary['custom_resnames']}")
    if summary["blank_chain_atoms"]:
        warnings.append(f"Blank chain IDs: {summary['blank_chain_atoms']} atoms")
    if summary["missing_elements"]:
        warnings.append(f"Missing element fields: {summary['missing_elements']} atoms")
    if not summary["link_records"] and not summary["conect_records"]:
        warnings.append("No LINK/CONECT records advertise the crosslink")
    if product_state_pablo_library is None:
        warnings.append("Product-state Pablo residue library is not available")
    return {**summary, "warnings": warnings or ["No text-level blockers detected"]}


pablo_preflight = inspect_selected_pdb_for_pablo(selected_product_pdb)
pprint(pablo_preflight)

In [ ]:
def describe_pablo_api() -> None:
    """Print selected Pablo call signatures without ingesting a PDB."""
    if importlib.util.find_spec("openff.pablo") is None:
        print("openff.pablo is not importable in this kernel")
        return
    try:
        import inspect

        from openff import pablo
    except Exception as exc:
        print(f"openff.pablo import/signature setup failed: {type(exc).__name__}: {exc}")
        return

    targets = {
        "topology_from_pdb": pablo.topology_from_pdb,
        "ResidueDefinition.from_smiles": pablo.ResidueDefinition.from_smiles,
        "CcdCache.with_": pablo.CcdCache.with_,
        "CcdCache.with_crosslink": pablo.CcdCache.with_crosslink,
    }
    for label, obj in targets.items():
        print(f"{label}{inspect.signature(obj)}")


if RUN_PABLO_SMOKE:
    describe_pablo_api()
else:
    print("Skipping Pablo API signature check. Set RUN_PABLO_SMOKE = True to inspect it.")


In [ ]:
pablo_product_topology = None
if RUN_PABLO:
    try:
        from openff.pablo import topology_from_pdb

        if selected_product_pdb is None:
            raise RuntimeError("Need the generated crosslinked PDB before Pablo ingestion")
        if pablo_crosslink_spec is None:
            raise RuntimeError("Need crosslink metadata from the resolved attachment plan")
        if product_state_pablo_library is None:
            raise RuntimeError(
                "Product-state Pablo residue definitions are not implemented; "
                "not running Pablo with placeholder definitions."
            )
        pablo_product_topology = topology_from_pdb(
            selected_product_pdb,
            residue_library=product_state_pablo_library.residue_library,
            format="PDB",
            use_canonical_names=False,
        )
        pprint(
            {
                "pablo_topology_type": type(pablo_product_topology).__name__,
                "n_atoms": pablo_product_topology.n_atoms,
                "n_molecules": pablo_product_topology.n_molecules,
            }
        )
    except Exception as exc:
        pablo_product_topology = None
        print(f"Pablo topology build blocked/failed: {type(exc).__name__}: {str(exc).splitlines()[0]}")
        if "pablo_preflight" in globals():
            pprint(pablo_preflight.get("warnings", []))
else:
    print("Skipping Pablo ingestion. Set RUN_PABLO = True only after product-state definitions exist.")

In [ ]:
interchange_result = None
if RUN_INTERCHANGE_HANDOFF:
    try:
        from polyzymd.builders.conjugation.parameterization import (
            create_interchange_from_pablo_topology,
        )

        if pablo_product_topology is None:
            raise RuntimeError("Need a Pablo topology before Interchange handoff")
        if charged_product_template is None:
            raise RuntimeError(
                "Need a non-empty explicit charged product template for "
                "charge_from_molecules=[template]"
            )
        interchange_result = create_interchange_from_pablo_topology(
            pablo_product_topology,
            settings=InterchangeParameterizationSettings(),
            charge_from_molecules=[charged_product_template],
        )
        pprint(interchange_result.model_dump(mode="json", exclude={"interchange"}))
    except Exception as exc:
        interchange_result = None
        print(f"Interchange handoff blocked/failed: {type(exc).__name__}: {str(exc).splitlines()[0]}")
        print("This is expected until product-state charges and custom FF coverage are supplied.")
else:
    print("Skipping Interchange handoff. Set RUN_INTERCHANGE_HANDOFF = True after Pablo and charges succeed.")

## 9. Future production integration boundary

`system_workflow.py` is the production orchestration boundary for config-driven runs. It is not executed as the notebook happy path unless it consumes the same generated artifacts and has real product-state Pablo definitions/charge templates. The generated-product cell above is the current executable POC boundary.

In [ ]:
import inspect

from polyzymd.builders.conjugation import system_workflow

for name in [
    "ConjugatedPolymerSystemSettings",
    "build_conjugated_polymer_system_from_config_path",
    "build_conjugated_polymer_system_from_config",
]:
    obj = getattr(system_workflow, name)
    print(f"\n{name}")
    print(obj.model_fields if inspect.isclass(obj) and hasattr(obj, "model_fields") else inspect.signature(obj))

settings = system_workflow.ConjugatedPolymerSystemSettings(create_final_interchange=False)
print("\nReference settings only; not part of this notebook's generated-product path:")
pprint(settings.model_dump(mode="json"))

if RUN_SYSTEM_WORKFLOW:
    print(
        "RUN_SYSTEM_WORKFLOW is intentionally not wired here. Use the production "
        "workflow after it consumes generated product artifacts and real "
        "product-state Pablo/charge templates."
    )
else:
    print("Skipping production workflow boundary inspection beyond signatures/settings.")

In [ ]:
future_working_path = [
    "Load SOURCE_PROTEIN_PDB and select chain A LYS 23 NZ",
    "Generate seeded random NHS-containing polymer PDB/SDF artifacts",
    "Resolve NHS-Lys attachment plan from generated atoms",
    "Place reactive polymer atom near lysine NZ with Packmol constraints",
    "Write new assembled product PDB with chain/element/connectivity metadata",
    "Generate product-state Pablo residue definitions from the emitted product graph",
    "Provide explicit charged product template for charge_from_molecules=[template]",
    "Create Interchange only after Pablo topology and charges are real",
]
for step in future_working_path:
    print(f"- {step}")

if RUN_MARIMO_POC:
    print(
        "The old Marimo POC is retained only for diagnostics and is not the "
        "truth source for this end-to-end path."
    )
    spec = importlib.util.spec_from_file_location("conjugation_poc_marimo", POC_DIR / "conjugation_poc.py")
    if spec is None or spec.loader is None:
        raise RuntimeError("Could not load Marimo POC module spec")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    outputs, defs = mod.app.run()
    print("Definitions exported by Marimo app:")
    print(sorted(defs))
else:
    print("Skipping old Marimo POC execution. Set RUN_MARIMO_POC = True only for diagnostics.")

## 10. Abstraction pressure/code-smell notes

Keep these areas explicit until the right abstractions are clear:

- Packmol constraints know bond-forming atoms and leaving atoms
- RDKit graph surgery must synchronize atom indices, coordinates, residue names, and bond metadata
- Product-state Pablo residue definitions must be generated from emitted product graphs, not placeholder molecules
- Hybrid charge merge patches local junction charges and reconciles formal-charge residuals
- Interchange handoff requires explicit non-empty charged product templates
- `system_workflow.py` is a useful production boundary, but notebook execution should stay tied to generated artifacts